In [1]:
!conda info


     active environment : spark_env
    active env location : E:\Anaconda\envs\spark_env
            shell level : 2
       user config file : C:\Users\tayal\.condarc
 populated config files : E:\Anaconda\.condarc
                          C:\Users\tayal\.condarc
          conda version : 25.5.1
    conda-build version : 25.5.0
         python version : 3.13.5.final.0
                 solver : libmamba (default)
       virtual packages : __archspec=1=x86_64_v3
                          __conda=25.5.1=0
                          __win=10.0.22631=0
       base environment : E:\Anaconda  (writable)
      conda av data dir : E:\Anaconda\etc\conda
  conda av metadata url : None
           channel URLs : https://repo.anaconda.com/pkgs/main/win-64
                          https://repo.anaconda.com/pkgs/main/noarch
                          https://repo.anaconda.com/pkgs/r/win-64
                          https://repo.anaconda.com/pkgs/r/noarch
                          https://repo.anaconda

## Imports

In [54]:
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, min, max
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [6]:
spark = SparkSession.builder.appName('BCP').getOrCreate()
spark

## Load the Dataset


In [10]:
df = spark.read.csv('heart.csv', header = True, inferSchema = True)
df.show()

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
| 63|  1|  3|     145| 233|  1|      0|    150|    0|    2.3|    0|  0|   1|     1|
| 37|  1|  2|     130| 250|  0|      1|    187|    0|    3.5|    0|  0|   2|     1|
| 41|  0|  1|     130| 204|  0|      0|    172|    0|    1.4|    2|  0|   2|     1|
| 56|  1|  1|     120| 236|  0|      1|    178|    0|    0.8|    2|  0|   2|     1|
| 57|  0|  0|     120| 354|  0|      1|    163|    1|    0.6|    2|  0|   2|     1|
| 57|  1|  0|     140| 192|  0|      1|    148|    0|    0.4|    1|  0|   1|     1|
| 56|  0|  1|     140| 294|  0|      0|    153|    0|    1.3|    1|  0|   2|     1|
| 44|  1|  1|     120| 263|  0|      1|    173|    0|    0.0|    2|  0|   3|     1|
| 52|  1|  2|     172| 199|  1|      1|    162|    0|    0.5|    2|  0|   3|

In [11]:
df.describe().show()

+-------+------------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+
|summary|               age|                sex|                cp|          trestbps|              chol|                fbs|          restecg|           thalach|              exang|           oldpeak|             slope|                ca|              thal|            target|
+-------+------------------+-------------------+------------------+------------------+------------------+-------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+
|  count|               303|                303|               303|               303|               303|                303|              303|               303|    

In [12]:
total_rows = df.count()
total_rows

303

In [13]:
len(df.columns)

14

In [14]:
check_df = df.groupBy('target').count()
check_df.show()

+------+-----+
|target|count|
+------+-----+
|     1|  165|
|     0|  138|
+------+-----+



In [15]:
per_check_df = check_df.withColumn('percentage', check_df['count']/total_rows*100)
per_check_df.show()

+------+-----+-----------------+
|target|count|       percentage|
+------+-----+-----------------+
|     1|  165|54.45544554455446|
|     0|  138|45.54455445544555|
+------+-----+-----------------+



In [22]:
df.groupBy('target').avg('age').show()

+------+-----------------+
|target|         avg(age)|
+------+-----------------+
|     1| 52.4969696969697|
|     0|56.60144927536232|
+------+-----------------+



In [24]:
df.groupBy('target','cp').count().show()

+------+---+-----+
|target| cp|count|
+------+---+-----+
|     1|  0|   39|
|     1|  2|   69|
|     1|  1|   41|
|     1|  3|   16|
|     0|  0|  104|
|     0|  1|    9|
|     0|  2|   18|
|     0|  3|    7|
+------+---+-----+



In [26]:
df.filter(df['target']==1).agg(avg('chol'),min('chol'),max('chol')).show()

+------------------+---------+---------+
|         avg(chol)|min(chol)|max(chol)|
+------------------+---------+---------+
|242.23030303030302|      126|      564|
+------------------+---------+---------+



In [27]:
hd = df.filter(df['target']==1)
hd.groupBy('sex').count().show()

+---+-----+
|sex|count|
+---+-----+
|  1|   93|
|  0|   72|
+---+-----+



In [29]:
df.groupBy('age').avg('thalach').show()

+---+------------------+
|age|      avg(thalach)|
+---+------------------+
| 65|           146.125|
| 53|             138.0|
| 34|             183.0|
| 76|             116.0|
| 44| 168.8181818181818|
| 47|             149.6|
| 52|167.23076923076923|
| 40|157.66666666666666|
| 57| 143.8235294117647|
| 54|            147.75|
| 48|166.28571428571428|
| 64|             133.0|
| 41|             164.7|
| 43|           154.875|
| 37|             178.5|
| 61|           145.125|
| 35|             160.5|
| 59|147.57142857142858|
| 55|           139.625|
| 39|            163.25|
+---+------------------+
only showing top 20 rows



In [34]:
df.filter((df['fbs']==1) & (df['target']==1)).count()

23

In [43]:
df.columns

['age',
 'sex',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalach',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal',
 'target']

## Model Building

In [45]:
assembler = VectorAssembler(inputCols=['age',
 'sex',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalach',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal'],
                          outputCol='features')
model_input = assembler.transform(df)
model_input.show(truncate = False)

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+------------------------------------------------------------+
|age|sex|cp |trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope|ca |thal|target|features                                                    |
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+------------------------------------------------------------+
|63 |1  |3  |145     |233 |1  |0      |150    |0    |2.3    |0    |0  |1   |1     |[63.0,1.0,3.0,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,1.0]|
|37 |1  |2  |130     |250 |0  |1      |187    |0    |3.5    |0    |0  |2   |1     |[37.0,1.0,2.0,130.0,250.0,0.0,1.0,187.0,0.0,3.5,0.0,0.0,2.0]|
|41 |0  |1  |130     |204 |0  |0      |172    |0    |1.4    |2    |0  |2   |1     |[41.0,0.0,1.0,130.0,204.0,0.0,0.0,172.0,0.0,1.4,2.0,0.0,2.0]|
|56 |1  |1  |120     |236 |0  |1      |178    |0    |0.8    |2    |0  |2   |1     |[56.0,1.0,1.0,120.0,236.0,0.0,1.0,178.0,0.0,0.8

In [46]:
model_select = model_input.select('features','target')
model_select.show()

+--------------------+------+
|            features|target|
+--------------------+------+
|[63.0,1.0,3.0,145...|     1|
|[37.0,1.0,2.0,130...|     1|
|[41.0,0.0,1.0,130...|     1|
|[56.0,1.0,1.0,120...|     1|
|[57.0,0.0,0.0,120...|     1|
|[57.0,1.0,0.0,140...|     1|
|[56.0,0.0,1.0,140...|     1|
|[44.0,1.0,1.0,120...|     1|
|[52.0,1.0,2.0,172...|     1|
|[57.0,1.0,2.0,150...|     1|
|[54.0,1.0,0.0,140...|     1|
|[48.0,0.0,2.0,130...|     1|
|[49.0,1.0,1.0,130...|     1|
|[64.0,1.0,3.0,110...|     1|
|[58.0,0.0,3.0,150...|     1|
|[50.0,0.0,2.0,120...|     1|
|[58.0,0.0,2.0,120...|     1|
|[66.0,0.0,3.0,150...|     1|
|[43.0,1.0,0.0,150...|     1|
|[69.0,0.0,3.0,140...|     1|
+--------------------+------+
only showing top 20 rows



In [47]:
train_df, test_df = model_select.randomSplit([0.7,0.3], seed = 50)

In [48]:
train_df.show()

+--------------------+------+
|            features|target|
+--------------------+------+
|(13,[0,1,3,4,7,10...|     0|
|(13,[0,1,3,4,7,10...|     1|
|(13,[0,1,3,4,7,10...|     1|
|(13,[0,1,3,4,7,10...|     1|
|(13,[0,2,3,4,7,10...|     1|
|(13,[0,2,3,4,7,10...|     1|
|(13,[0,2,3,4,7,10...|     1|
|(13,[0,3,4,6,7,10...|     1|
|(13,[0,3,4,6,7,10...|     1|
|(13,[0,3,4,7,9,10...|     1|
|(13,[0,3,4,7,9,10...|     1|
|(13,[0,3,4,7,9,10...|     1|
|(13,[0,3,4,7,9,10...|     1|
|(13,[0,3,4,7,9,11...|     0|
|(13,[0,3,4,7,10,1...|     1|
|(13,[0,3,4,7,10,1...|     1|
|(13,[0,3,4,7,10,1...|     0|
|[29.0,1.0,1.0,130...|     1|
|[34.0,0.0,1.0,118...|     1|
|[34.0,1.0,3.0,118...|     1|
+--------------------+------+
only showing top 20 rows



In [49]:
test_df.show()

+--------------------+------+
|            features|target|
+--------------------+------+
|(13,[0,3,4,7,8,10...|     1|
|(13,[0,3,4,7,9,11...|     0|
|(13,[0,3,4,7,10,1...|     1|
|[35.0,0.0,0.0,138...|     1|
|[38.0,1.0,2.0,138...|     1|
|[38.0,1.0,3.0,120...|     0|
|[40.0,1.0,0.0,110...|     0|
|[40.0,1.0,0.0,152...|     0|
|[40.0,1.0,3.0,140...|     1|
|[41.0,0.0,2.0,112...|     1|
|[41.0,1.0,1.0,120...|     1|
|[41.0,1.0,2.0,130...|     1|
|[42.0,1.0,2.0,120...|     1|
|[43.0,1.0,0.0,150...|     1|
|[44.0,1.0,0.0,110...|     0|
|[44.0,1.0,0.0,120...|     0|
|[44.0,1.0,2.0,120...|     1|
|[45.0,0.0,1.0,112...|     1|
|[46.0,0.0,1.0,105...|     1|
|[46.0,0.0,2.0,142...|     1|
+--------------------+------+
only showing top 20 rows



## Logistic Regression

In [65]:
lr = LogisticRegression(featuresCol='features', labelCol='target', maxIter=100)
model_lr = lr.fit(train_df)
model_train_prediction_lr = model_lr.transform(train_df)
model_test_prediction_lr = model_lr.transform(test_df)

## Decision Tree

In [66]:
dt = DecisionTreeClassifier(featuresCol='features', labelCol='target', maxDepth=3)
model_dt = dt.fit(train_df)
model_train_prediction_dt = model_dt.transform(train_df)
model_test_prediction_dt = model_dt.transform(test_df)

## Random Forest

In [77]:
rf = RandomForestClassifier(featuresCol='features', labelCol='target', maxDepth=3,numTrees=300)
model_rf = rf.fit(train_df)
model_train_prediction_rf = model_rf.transform(train_df)
model_test_prediction_rf = model_rf.transform(test_df)

## Evaluation

In [78]:
acc = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='target',metricName='accuracy')
f1 = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='target',metricName='f1')
pre = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='target',metricName='precisionByLabel')
rec = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='target',metricName='recallByLabel')

In [79]:
def eval(model_predictions):
    print('Accuracy = ',acc.evaluate(model_predictions))
    print('f1 Score = ',f1.evaluate(model_predictions))
    print('Precision = ',pre.evaluate(model_predictions))
    print('Recall = ',rec.evaluate(model_predictions))
    model_predictions.groupBy('target','prediction').count().show()

## Logistic Regression

In [70]:
print('Training')
eval(model_train_prediction_lr)
print('Testing')
eval(model_test_prediction_lr)

Training
Accuracy =  0.8809523809523809
f1 Score =  0.8804336011225407
Precision =  0.8977272727272727
Recall =  0.8315789473684211
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|    9|
|     0|       0.0|   79|
|     1|       1.0|  106|
|     0|       1.0|   16|
+------+----------+-----+

Testing
Accuracy =  0.7634408602150538
f1 Score =  0.7629438872323123
Precision =  0.7560975609756098
Recall =  0.7209302325581395
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|   10|
|     0|       0.0|   31|
|     1|       1.0|   40|
|     0|       1.0|   12|
+------+----------+-----+



## Decision Tree

In [71]:
print('Training')
eval(model_train_prediction_dt)
print('Testing')
eval(model_test_prediction_dt)

Training
Accuracy =  0.8809523809523809
f1 Score =  0.8800113475928862
Precision =  0.9166666666666666
Recall =  0.8105263157894737
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|    7|
|     0|       0.0|   77|
|     1|       1.0|  108|
|     0|       1.0|   18|
+------+----------+-----+

Testing
Accuracy =  0.7849462365591398
f1 Score =  0.7805731337140506
Precision =  0.8484848484848485
Recall =  0.6511627906976745
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|    5|
|     0|       0.0|   28|
|     1|       1.0|   45|
|     0|       1.0|   15|
+------+----------+-----+



## Random Forest

In [80]:
print('Training')
eval(model_train_prediction_rf)
print('Testing')
eval(model_test_prediction_rf)

Training
Accuracy =  0.8904761904761904
f1 Score =  0.8893837789004142
Precision =  0.9390243902439024
Recall =  0.8105263157894737
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|    5|
|     0|       0.0|   77|
|     1|       1.0|  110|
|     0|       1.0|   18|
+------+----------+-----+

Testing
Accuracy =  0.7849462365591398
f1 Score =  0.7829681476973017
Precision =  0.8108108108108109
Recall =  0.6976744186046512
+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     1|       0.0|    7|
|     0|       0.0|   30|
|     1|       1.0|   43|
|     0|       1.0|   13|
+------+----------+-----+

